[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/13_gpt2_block.ipynb)

# 🔴 Hard: GPT-2 Transformer Block

Implement a full **GPT-2 style Transformer block** — combining everything you've learned.

### Architecture (Pre-Norm)
```
x = x + causal_self_attention(ln1(x))
x = x + mlp(ln2(x))
```

### Signature
```python
class GPT2Block(nn.Module):
    def __init__(self, d_model: int, num_heads: int): ...
    def forward(self, x: torch.Tensor) -> torch.Tensor: ...
```

### Requirements
- Inherit from `nn.Module`
- `self.ln1`, `self.ln2`: `nn.LayerNorm(d_model)`
- `self.W_q`, `self.W_k`, `self.W_v`, `self.W_o`: `nn.Linear` for attention
- `self.mlp`: `nn.Sequential(Linear(d, 4d), GELU(), Linear(4d, d))`
- Attention must be **causal** (mask future positions)
- Pre-norm architecture (LayerNorm *before* attention and MLP)
- Residual connections around both attention and MLP

In [2]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [3]:
import torch
import torch.nn as nn
import math

/Users/yaoliu/Documents/TorchCode/.venv/lib/python3.11/site-packages/torch/_subclasses/functional_tensor.py:307: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


In [16]:
# ✏️ YOUR IMPLEMENTATION HERE

class GPT2Block(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.d_model = d_model 
        self.num_heads = num_heads

        self.d_k = d_model // num_heads
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

        self.mlp = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Linear(4 * d_model, d_model)
        )
    
    def casual_self_attention(self, x):
        B, S, _ = x.shape
        q = self.W_q(x).view(B, S, self.num_heads, self.d_k).transpose(1, 2)
        k = self.W_k(x).view(B, S, self.num_heads, self.d_k).transpose(1, 2)
        v = self.W_v(x).view(B, S, self.num_heads, self.d_k).transpose(1, 2)

        scores = torch.matmul(q, k.transpose(-2, -1)) / (self.d_k ** 0.5)
        mask = torch.triu(torch.ones(S, S, device=x.device), diagonal=1).bool()
        scores = scores.masked_fill(mask, float('-inf'))

        attn = torch.softmax(scores, -1)
        out = torch.matmul(attn, v).transpose(1, 2).reshape(B, S, -1)
        return self.W_o(out)

    def forward(self, x):
        # Pre-norm + causal attention + MLP with residuals
        x = x + self.casual_self_attention(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x
                

In [17]:
# 🧪 Debug
torch.manual_seed(0)
block = GPT2Block(d_model=64, num_heads=4)
x = torch.randn(2, 8, 64)
out = block(x)
print("Output shape:", out.shape)           # (2, 8, 64)
print("Is nn.Module?", isinstance(block, nn.Module))
print("Params:", sum(p.numel() for p in block.parameters()))

Output shape: torch.Size([2, 8, 64])
Is nn.Module? True
Params: 49984


In [18]:
from torch_judge import check
check('gpt2_block')


🧪 Testing: GPT-2 Transformer Block (Hard)
──────────────────────────────────────────────────
  ✅ [1/5] Output shape (3.2ms)
  ✅ [2/5] Has LayerNorm (pre-norm architecture) (0.3ms)
  ✅ [3/5] MLP has 4x expansion with GELU (0.2ms)
  ✅ [4/5] Causal masking — future doesn't affect past (1.4ms)
  ✅ [5/5] Gradient flow to all parameters (18.3ms)
──────────────────────────────────────────────────
  🎉 All 5 tests passed! (23.4ms total)
  Progress saved. Run status() to see your dashboard.

